# 01 - Raw Data Acquisition

Main purpose: Pull all the raw data necessary for the later steps. 

In [49]:
import pandas as pd
import numpy as np
import yfinance as yf
import os
from fredapi import Fred

In [50]:
# SPX returns
#Pulled data from 1990 so that we can have complete rolling windows from 1996 onwards
spx_raw = yf.download('^GSPC', start = '1990-01-01', multi_level_index = False).reset_index() 
spx_raw['Returns'] = spx_raw['Close'].pct_change()
spx_raw = spx_raw.dropna()
spx_raw.to_csv('data/raw/01_spx_returns.csv', index=False)
spx_raw

[*********************100%***********************]  1 of 1 completed


,Date,Close,High,Low,Open,Volume,Returns
1,1990-01-03,358.760010,360.589996,357.890015,359.690002,192330000,-0.002586
2,1990-01-04,355.670013,358.760010,352.890015,358.760010,177000000,-0.008613
3,1990-01-05,352.200012,355.670013,351.350006,355.670013,158530000,-0.009756
4,1990-01-08,353.790009,354.239990,350.540009,352.200012,140110000,0.004514
5,1990-01-09,349.619995,354.170013,349.609985,353.829987,155210000,-0.011787
...,...,...,...,...,...,...,...
9169,2026-06-01,7599.959961,7617.660156,7562.609863,7582.290039,6583960000,0.002625
9170,2026-06-02,7609.779785,7620.899902,7582.990234,7595.399902,5904410000,0.001292
9171,2026-06-03,7553.680176,7605.350098,7551.220215,7605.310059,5753860000,-0.007372
9172,2026-06-04,7584.310059,7598.189941,7516.540039,7516.540039,5398910000,0.004055


In [51]:
FRED_API_KEY = '0a07242543300b5582000574843752d6'
fred = Fred(api_key=FRED_API_KEY)

In [52]:
# FRED data
series = {
    #credit data
    'BAA': 'DBAA',
    'AAA': 'DAAA',
    'HY_OAS': 'BAMLH0A0HYM2',
    'IG_OAS': 'BAMLC0A0CM',
    
    #funding/financial conditions
    'TED': 'TEDRATE',
    'SOFR': 'SOFR',

    # Section F - Treasury yields
    'DGS3MO': 'DGS3MO',
    'DGS2': 'DGS2',
    'DGS10': 'DGS10',
    'DTB3': 'DTB3'
}

frames = []
#Pulled data from 1990 so that we can have complete rolling windows from 1996 onwards
for ID in series.values():
    frame = fred.get_series(ID, observation_start = '1990-01-01')
    frames.append(frame)

fred_raw = pd.concat(frames, axis = 1).reset_index()
fred_raw.columns = ['Date'] + list(series.keys())
fred_raw = fred_raw.ffill()
fred_raw.to_csv('data/raw/02_fred_raw.csv', index=False)
fred_raw

/var/folders/3f/c9b7x0c96rd39cz0kp8cc_nr0000gn/T/ipykernel_8288/3925364451.py:26: Pandas4Warning: Sorting by default when concatenating all DatetimeIndex is deprecated.  In the future, pandas will respect the default of `sort=False`. Specify `sort=True` or `sort=False` to silence this message. If you see this warnings when not directly calling concat, report a bug to pandas.
  fred_raw = pd.concat(frames, axis = 1).reset_index()


,Date,BAA,AAA,HY_OAS,IG_OAS,TED,SOFR,DGS3MO,DGS2,DGS10,DTB3
0,1990-01-01,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,1990-01-02,9.85,8.87,NaN,NaN,0.80,NaN,7.83,7.87,7.94,7.58
2,1990-01-03,9.86,8.90,NaN,NaN,0.75,NaN,7.89,7.94,7.99,7.63
3,1990-01-04,9.84,8.89,NaN,NaN,0.79,NaN,7.84,7.92,7.98,7.59
4,1990-01-05,9.88,8.88,NaN,NaN,0.84,NaN,7.79,7.90,7.99,7.54
...,...,...,...,...,...,...,...,...,...,...,...
9511,2026-05-31,6.02,5.50,2.74,0.74,0.09,3.63,3.69,3.98,4.45,3.60
9512,2026-06-01,6.01,5.49,2.72,0.73,0.09,3.65,3.78,4.05,4.47,3.63
9513,2026-06-02,6.00,5.48,2.71,0.74,0.09,3.63,3.77,4.05,4.46,3.63
9514,2026-06-03,6.03,5.51,2.75,0.74,0.09,3.61,3.78,4.08,4.49,3.63


In [53]:
#yfinance data

series = {
    'HYG': 'HYG',
    'LQD': 'LQD',
    'XLF': 'XLF',
    'WTI': 'CL=F',
    'USD': 'DX-Y.NYB',
    'GOLD': 'GC=F'
}

frames = []
#Pulled data from 1990 so that we can have complete rolling windows from 1996 onwards
for ID in series.values():
    frame = yf.download(ID, start = '1990-01-01', multi_level_index = False)['Close']
    frames.append(frame)

yf_raw = pd.concat(frames, axis = 1).reset_index()
yf_raw.columns = ['Date'] + list(series.keys())
yf_raw['WTI']  = yf_raw['WTI'].ffill()
yf_raw['GOLD'] = yf_raw['GOLD'].ffill()
yf_raw['USD']  = yf_raw['USD'].ffill()
yf_raw.to_csv('data/raw/03_yf_raw.csv', index=False)
yf_raw
    

[*********************100%***********************]  1 of 1 completed
[*********************100%***********************]  1 of 1 completed
[*********************100%***********************]  1 of 1 completed
[*********************100%***********************]  1 of 1 completed
[*********************100%***********************]  1 of 1 completed
[*********************100%***********************]  1 of 1 completed
/var/folders/3f/c9b7x0c96rd39cz0kp8cc_nr0000gn/T/ipykernel_8288/625190485.py:18: Pandas4Warning: Sorting by default when concatenating all DatetimeIndex is deprecated.  In the future, pandas will respect the default of `sort=False`. Specify `sort=True` or `sort=False` to silence this message. If you see this warnings when not directly calling concat, report a bug to pandas.
  yf_raw = pd.concat(frames, axis = 1).reset_index()


,Date,HYG,LQD,XLF,WTI,USD,GOLD
0,1990-01-01,NaN,NaN,NaN,NaN,93.209999,NaN
1,1990-01-02,NaN,NaN,NaN,NaN,94.290001,NaN
2,1990-01-03,NaN,NaN,NaN,NaN,94.419998,NaN
3,1990-01-04,NaN,NaN,NaN,NaN,92.519997,NaN
4,1990-01-05,NaN,NaN,NaN,NaN,92.849998,NaN
...,...,...,...,...,...,...,...
9284,2026-06-02,79.900002,108.919998,51.459999,93.760002,99.220001,4489.100098
9285,2026-06-03,79.680000,108.620003,50.869999,96.019997,99.529999,4436.700195
9286,2026-06-04,79.830002,108.849998,52.189999,93.040001,99.410004,4475.799805
9287,2026-06-05,79.430000,108.169998,52.299999,90.540001,100.070000,4337.100098
